# STIRPAT과 로그회귀

## 국가 패널자료에서 환경영향의 탄력성 추정

관측값과 회귀선의 차이인 잔차, 잔차제곱합을 최소화하는 OLS의 원리를 먼저 확인합니다. 이 기초 위에서 인구와 경제수준이 CO₂ 배출과 어떻게 연관되는지 로그–로그 회귀로 요약하고, pooled OLS와 국가·연도 고정효과 명세의 계수를 비교합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 회귀식의 결과변수·설명변수·계수·적합값·잔차를 구분하고, 잔차제곱합(SSE)을 이용해 회귀선을 비교합니다.
- 횡단면·시계열·국가–연도 패널자료의 차이를 설명합니다.
- IPAT 항등식과 STIRPAT 확률모형의 차이를 설명합니다.
- 로그–로그 회귀계수를 탄력성으로 해석합니다.
- 모든 국가–연도 행을 모은 pooled OLS를 기준선으로 적합하고 국가·연도 고정효과 결과와 비교합니다.
- `statsmodels`의 회귀 추론 결과와 `scikit-learn`의 예측 절차가 제공하는 정보를 구분합니다.
- 계수와 표준오차의 역할을 구분하고 인과해석의 한계를 기록합니다.


## 이번 교시에서 사용할 데이터

### STIRPAT 수업용 국가 패널

| 항목 | 내용 |
|---|---|
| 구성 | 한국을 포함한 12개국 |
| 기간과 규모 | 1990~2022년, 396행의 균형 패널 |
| 한 행의 의미 | 한 국가의 한 연도 환경·인구·경제 관측 |
| 결과변수 | `co2_tonnes` |
| 설명변수 | `population`, `gdp_per_capita_constant_2021_intl` |
| provenance 기록 | `gdp_value_status` |

이 자료는 공개 지표를 수업 명세에 맞춰 정리한 snapshot이며 York 등의 원 논문 표본을 복제한 자료가 아닙니다. 금액 변수는 constant 2021 international dollars 기준이므로 명목 GDP와 혼용하지 않습니다.

이 자료에는 같은 국가가 여러 연도에 반복됩니다. 따라서 국가가 다른 행과 같은 국가의 연도가 다른 행을 구분해서 읽어야 합니다. 강의에서는 12개국 전체를 사용하고, 적용 실습에서는 국가 목록 또는 기간 하나만 변경합니다. 회귀명세와 표준오차 조정은 자료의 행 구조를 확인한 뒤 단계적으로 소개합니다.


## 회귀 논문을 Python 객체로 읽는 순서

회귀는 여러 관측에서 설명변수와 결과변수가 함께 달라지는 관계를 하나의 식으로 요약합니다.

$$y_i=\beta_0+\beta_1x_i+e_i$$

| 개념 | Python에서 확인할 값 | 의미 |
|---|---|---|
| 관측값 | `y` | 실제로 측정한 결과변수 |
| 적합값 | `fitted` | 추정된 회귀선이 계산한 값 |
| 잔차 | `residual = y - fitted` | 현재 회귀식이 설명하지 못한 차이 |
| 계수 | `coef` | 설명변수와 결과변수 관계의 방향과 크기 |

이번 교시는 회귀 용어를 한꺼번에 암기하지 않습니다. 산점도에서 관계를 보고, 회귀선을 적합하고, 잔차를 확인한 뒤 패널자료의 비교 방식으로 확장합니다.


## 산점도와 회귀선으로 관계의 형태 확인하기

산점도에서 점 하나는 관측 한 건입니다. x가 증가할수록 y도 증가하는지, 관계가 직선에 가까운지, 일부 점이 전체 기울기를 크게 바꾸는지 먼저 봅니다.

회귀선은 모든 점을 통과하는 선이 아니라 자료의 평균적 방향을 요약하는 선입니다. 기울기가 양수이면 x가 큰 관측에서 y도 큰 경향, 음수이면 반대 경향을 나타냅니다.

다음 실행 셀에서는 작은 자료표를 출력한 뒤 산점도와 추정 회귀선을 같은 그림에 표시합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sample = pd.DataFrame(
    {
        "energy_use": [2.0, 2.8, 3.6, 4.3, 5.2, 6.1, 7.0],
        "co2": [1.5, 2.2, 2.4, 3.7, 3.8, 5.2, 5.5],
    }
)
slope, intercept = np.polyfit(sample["energy_use"], sample["co2"], deg=1)
line_x = np.linspace(sample["energy_use"].min(), sample["energy_use"].max(), 100)

print("[회귀선에 사용할 작은 자료]")
display(sample)
print(f"추정식: CO₂ = {intercept:.3f} + {slope:.3f} × 에너지사용량")
print("기울기는 에너지사용량이 1단위 큰 관측에서 CO₂가 평균적으로 얼마나 다른지를 나타냅니다.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.scatter(sample["energy_use"], sample["co2"], s=55, color="#4C78A8", label="observed")
ax.plot(line_x, intercept + slope * line_x, color="#E45756", linewidth=2, label="OLS fitted line")
ax.set(title="Scatter plot and fitted regression line", xlabel="Energy use", ylabel="CO₂")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 적합값·잔차·SSE로 OLS 결과 확인하기

OLS는 **Ordinary Least Squares**, 한국어로 보통최소제곱법입니다. 여러 후보 직선 가운데 잔차 제곱합이 가장 작은 절편과 기울기를 선택합니다.

$$e_i=y_i-\hat y_i, \qquad
SSE=\sum_i e_i^2$$

잔차를 제곱하면 양수와 음수가 서로 상쇄되지 않고, 큰 차이에 더 큰 부담을 줍니다. 잔차는 잘못된 데이터라는 뜻이 아니라 현재 모형이 설명하지 못하고 남긴 차이입니다.

다음 실행 셀에서는 각 행의 실제값·적합값·잔차·잔차제곱을 표로 확인하고, 잔차가 0선 주변에 어떻게 놓이는지 그립니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sample = pd.DataFrame(
    {
        "x": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
        "observed_y": [2.0, 2.7, 4.6, 4.9, 7.2, 7.8],
    }
)
design = np.column_stack([np.ones(len(sample)), sample["x"]])
intercept, slope = np.linalg.lstsq(design, sample["observed_y"], rcond=None)[0]
sample["fitted"] = intercept + slope * sample["x"]
sample["residual"] = sample["observed_y"] - sample["fitted"]
sample["residual_squared"] = sample["residual"] ** 2
sse = sample["residual_squared"].sum()

print("[행별 적합값과 잔차]")
display(sample.round(3))
print(f"절편 = {intercept:.3f}, 기울기 = {slope:.3f}")
print(f"SSE = 잔차제곱의 합 = {sse:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.scatter(sample["fitted"], sample["residual"], color="#4C78A8", s=55)
ax.axhline(0, color="#E45756", linewidth=1.2)
for row_number, row in sample.iterrows():
    ax.annotate(str(row_number), (row["fitted"], row["residual"]), xytext=(4, 4), textcoords="offset points")
ax.set(title="Residuals versus fitted values", xlabel="Fitted value", ylabel="Residual = observed - fitted")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 횡단면·시계열·패널은 행의 반복 방식이 다르다

| 자료 구조 | 행의 예 | 주로 포함하는 비교 |
|---|---|---|
| 횡단면 | 같은 해의 한국·일본·미국 | 대상 사이의 차이 |
| 시계열 | 한국의 여러 연도 | 한 대상 안의 시간 변화 |
| 패널 | 여러 국가의 여러 연도 | 국가 간 차이와 국가 내 변화가 함께 존재 |

패널자료에서 `KOR–2000`, `KOR–2020`, `JPN–2020`은 서로 다른 행입니다. 그러나 같은 국가의 여러 연도는 지리, 제도, 기본 산업구조처럼 공유하는 특성이 있어 완전히 무관한 관측으로 보기 어렵습니다.


## pooled OLS가 섞는 두 종류의 비교

`pooled`는 여러 국가와 연도 행을 한 표에 모은다는 뜻입니다. pooled OLS는 전체 행에 하나의 공통 절편과 공통 계수를 적합합니다.

$$\ln(CO_{2,it})=\beta_0+\beta_1\ln(P_{it})+\beta_2\ln(A_{it})+e_{it}$$

이 계수에는 두 종류의 정보가 함께 들어갑니다.

- 한국과 미국처럼 **서로 다른 국가 사이의 수준 차이**
- 한국의 2000년과 2020년처럼 **같은 국가 안의 시간 변화**

pooled OLS는 최종 정답이 아니라 가장 단순한 기준선입니다. 이후 국가별 기본 차이를 제거했을 때 계수가 얼마나 달라지는지 비교합니다.


## pooled 계수와 국가내 계수를 분리해 비교하기

국가마다 오랫동안 유지되는 기본 수준이 크면 pooled 계수는 국가 간 차이에 강하게 영향을 받을 수 있습니다. 각 변수에서 국가별 평균을 빼면 같은 국가가 자기 평균보다 높거나 낮았던 정도를 비교할 수 있습니다.

$$
x^{within}_{it}=x_{it}-\bar{x}_i, \qquad
y^{within}_{it}=y_{it}-\bar{y}_i
$$

이 변환은 국가 고정효과의 핵심 직관입니다. 다음 실행 셀에서는 같은 패널자료로 pooled 기울기와 국가내 기울기를 각각 계산해 두 비교가 왜 다른 답을 줄 수 있는지 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

panel = pd.DataFrame(
    {
        "country": ["A"] * 4 + ["B"] * 4 + ["C"] * 4,
        "year": [2019, 2020, 2021, 2022] * 3,
        "x": [1.0, 2.0, 3.0, 4.0, 6.0, 7.0, 8.0, 9.0, 11.0, 12.0, 13.0, 14.0],
        "y": [11.0, 10.6, 10.2, 9.8, 20.0, 19.6, 19.2, 18.8, 29.0, 28.6, 28.2, 27.8],
    }
)

pooled_design = np.column_stack([np.ones(len(panel)), panel["x"]])
pooled_intercept, pooled_slope = np.linalg.lstsq(pooled_design, panel["y"], rcond=None)[0]

panel["x_within"] = panel["x"] - panel.groupby("country")["x"].transform("mean")
panel["y_within"] = panel["y"] - panel.groupby("country")["y"].transform("mean")
within_slope = np.dot(panel["x_within"], panel["y_within"]) / np.dot(panel["x_within"], panel["x_within"])

coefficients = pd.DataFrame(
    {
        "계수": ["pooled 기울기", "국가내 기울기"],
        "추정값": [pooled_slope, within_slope],
        "주로 사용하는 비교": ["국가 간 수준 차이 + 국가 내 변화", "각 국가의 자기 평균 대비 변화"],
    }
)

print("[국가–연도 패널과 국가내 편차]")
display(panel.round(2))
print("\n[같은 자료에서 얻은 두 기울기]")
display(coefficients.round(3))
print("국가 간 수준 차이는 양의 방향이지만, 각 국가 안에서는 x가 늘 때 y가 감소하도록 만든 예입니다.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
for country, part in panel.groupby("country"):
    axes[0].plot(part["x"], part["y"], marker="o", label=f"Country {country}")
pooled_x = np.linspace(panel["x"].min(), panel["x"].max(), 100)
axes[0].plot(pooled_x, pooled_intercept + pooled_slope * pooled_x, color="black", linestyle="--", label="pooled line")
axes[0].set(title="Raw panel: between and within variation", xlabel="x", ylabel="y")
axes[0].legend(ncol=2, fontsize=8)

axes[1].scatter(panel["x_within"], panel["y_within"], color="#4C78A8")
within_x = np.linspace(panel["x_within"].min(), panel["x_within"].max(), 100)
axes[1].plot(within_x, within_slope * within_x, color="#E45756", linewidth=2)
axes[1].axhline(0, color="grey", linewidth=0.7)
axes[1].axvline(0, color="grey", linewidth=0.7)
axes[1].set(title="Within-country variation after demeaning", xlabel="x - country mean(x)", ylabel="y - country mean(y)")
plt.tight_layout()
plt.show()


## STIRPAT을 로그회귀 명세로 번역하기

IPAT는 환경영향을 인구·풍요·기술의 곱으로 표현합니다. STIRPAT는 이 관계의 크기를 자료에서 추정할 수 있도록 확률적 회귀모형으로 확장합니다.

$$I=aP^bA^c e$$

양변에 로그를 취하면 곱셈 관계가 덧셈형 회귀가 됩니다.

$$\ln I=\ln a+b\ln P+c\ln A+\varepsilon$$

이 수업에서 $I$는 CO₂ 배출량, $P$는 인구, $A$는 1인당 GDP입니다. 원 논문의 결과를 복제하는 것이 아니라, York, Rosa, and Dietz (2003)가 제안한 모형 구조를 축소 패널자료에서 구현하고 명세에 따른 계수 변화를 읽습니다.

출처: [York, Rosa, and Dietz (2003)](https://doi.org/10.1016/S0921-8009(03)00188-5)


## log–log 계수를 탄력성으로 계산하고 읽기

결과변수와 설명변수를 모두 로그로 바꾸면 계수는 퍼센트 변화 사이의 관계인 탄력성으로 읽습니다.

예를 들어 인구 계수 $b=0.8$이라면 다음처럼 해석합니다.

> 다른 모형 항이 같을 때 인구가 1% 큰 관측은 CO₂ 배출량이 약 0.8% 큰 것과 연관된다.

이는 0.8톤 증가라는 뜻이 아닙니다. 작은 변화에서는 계수 자체를 근사 탄력성으로 읽고, 정확한 1% 변화는 $100\{\exp[b\ln(1.01)]-1\}$로 계산할 수 있습니다.

다음 실행 셀에서는 양수 원자료를 로그로 변환하고, 두 탄력성 계수와 1% 변화 시나리오를 표와 그림으로 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sample = pd.DataFrame(
    {
        "observation": list("ABCDEFGH"),
        "population": [8, 10, 12, 15, 18, 22, 27, 33],
        "gdp_per_capita": [20, 28, 24, 35, 31, 42, 38, 50],
    }
)
deterministic_error = np.array([0.00, 0.03, -0.02, 0.01, -0.03, 0.02, -0.01, 0.00])
sample["co2"] = np.exp(
    -1.2
    + 0.72 * np.log(sample["population"])
    + 0.88 * np.log(sample["gdp_per_capita"])
    + deterministic_error
)

for column in ["population", "gdp_per_capita", "co2"]:
    if (sample[column] <= 0).any():
        raise ValueError(f"{column}에는 로그변환할 수 없는 0 이하 값이 있습니다.")
    sample[f"ln_{column}"] = np.log(sample[column])

design = np.column_stack(
    [np.ones(len(sample)), sample["ln_population"], sample["ln_gdp_per_capita"]]
)
intercept, beta_population, beta_affluence = np.linalg.lstsq(
    design, sample["ln_co2"], rcond=None
)[0]

coefficient_table = pd.DataFrame(
    {
        "설명변수": ["ln(population)", "ln(gdp_per_capita)"],
        "탄력성 계수": [beta_population, beta_affluence],
        "설명변수 1% 증가 시 CO₂ 정확 변화율(%)": [
            100 * (np.exp(beta_population * np.log(1.01)) - 1),
            100 * (np.exp(beta_affluence * np.log(1.01)) - 1),
        ],
    }
)

print("[양수 원자료와 로그변환 열]")
display(sample.round(3))
print("\n[log–log 회귀의 탄력성]")
display(coefficient_table.round(3))
print(f"추정식: ln(CO₂) = {intercept:.3f} + {beta_population:.3f} ln(P) + {beta_affluence:.3f} ln(A)")
print("계수는 톤이 아니라 1% 변화와 연결되는 CO₂의 퍼센트 변화입니다.")

coefficient_plot = coefficient_table.assign(
    plot_label=["ln(population)", "ln(GDP per capita)"]
)
ax = coefficient_plot.set_index("plot_label")["탄력성 계수"].plot.bar(
    figsize=(7.5, 3.4), color=["#4C78A8", "#F58518"]
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Estimated log–log elasticities", xlabel="", ylabel="Elasticity coefficient")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 고정효과·불확실성·인과해석의 범위

국가 고정효과는 관측기간 동안 거의 변하지 않는 국가별 기본 차이를, 연도 고정효과는 같은 해 여러 국가가 공유한 충격을 분리하는 데 사용됩니다. 그러나 시간에 따라 변하는 누락요인, 역인과, 측정오차까지 자동으로 해결하지는 않습니다.

계수는 관계의 방향과 크기를 요약하고, 표준오차는 같은 절차를 반복했을 때 계수가 얼마나 흔들릴 수 있는지를 나타냅니다. 두 값의 역할을 구분해 읽습니다.

최종 보고에는 다음을 함께 제시합니다.

`표본과 기간 + 자료 구조 + 모형식 + 계수와 단위 + pooled/고정효과 비교 + 불확실성 + 인과해석 한계`

회귀 결과표의 한 계수만 떼어내기보다, 그 계수가 어떤 행과 어떤 비교에서 계산됐는지를 추적하는 것이 이번 교시의 핵심입니다.


## 실행 환경과 입력자료

| 재현 조건 | 확인할 출력 |
|---|---|
| 입력자료 | 실제 파일 경로와 파일명 |
| 자료 구조 | 데이터 소개에 제시한 예상 `shape`와 필수 열 |
| 실행환경 | Python과 주요 라이브러리 버전 |

다음 셀은 분석에 사용한 파일과 실행환경을 기록합니다. 파일 경로 또는 `shape`가 강의 기준과 다르면 입력 파일과 버전을 확인한 뒤 분석을 진행합니다.

> **실행 전 확인:** 이번 분석의 입력 파일과 예상 행 수는 무엇인가요?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 두 후보 회귀선의 잔차제곱합 비교

같은 네 관측에 두 후보선을 적용합니다. 각 행의 잔차와 잔차제곱을 계산하고, 어느 선의 SSE가 작은지 확인하세요.


In [ ]:
regression_example = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0], "actual_y": [3.0, 5.0, 6.0, 10.0]})
candidate_lines = {
    "line_A: 1 + 2x": 1 + 2 * regression_example["x"],
    "line_B: 2 + 1.5x": 2 + 1.5 * regression_example["x"],
}
sse_rows = []
for name, prediction in candidate_lines.items():
    regression_example[f"{name} prediction"] = prediction
    regression_example[f"{name} residual"] = regression_example["actual_y"] - prediction
    regression_example[f"{name} residual_squared"] = regression_example[f"{name} residual"] ** 2
    sse_rows.append({"candidate": name, "SSE": regression_example[f"{name} residual_squared"].sum()})

display(regression_example.round(2))
sse_table = pd.DataFrame(sse_rows).sort_values("SSE")
print("[같은 네 행에서 두 후보선의 SSE]")
display(sse_table)
print("OLS는 가능한 계수 가운데 train SSE가 가장 작은 조합을 찾습니다.")


## 1. 패널자료의 로그변수 생성

로그변환 전에 grain, key와 양수 조건을 확인합니다.

| 원 변수 | 로그 변수 | 유지되는 정보 |
|---|---|---|
| 총 CO₂ 배출량 | `ln_co2` | 국가–연도 관측과 key |
| 인구 | `ln_population` | 국가와 연도 |
| 1인당 GDP | `ln_affluence` | 표본과 지표 출처 |

로그변환은 STIRPAT의 곱셈 관계를 덧셈형 회귀식으로 표현하고 계수를 탄력성으로 해석하기 위한 단계입니다. 0과 음수 값은 로그 정의역에 포함되지 않으므로 원자료에서 확인합니다.

> **확인 질문:** 로그변환 뒤에도 행 수와 `code + year` key가 유지되어야 하는 이유는 무엇입니까?


In [ ]:
import statsmodels.formula.api as smf

PANEL_PATH = find_course_data("methods/stirpat_panel.csv")
panel = pd.read_csv(PANEL_PATH)
SELECTED_CODES = ['AUS', 'BRA', 'CAN', 'CHN', 'FRA', 'DEU', 'IND', 'JPN', 'ZAF', 'KOR', 'GBR', 'USA']
START_YEAR = 1990
END_YEAR = 2022
analysis_name = "STIRPAT 12-country panel"

analysis = (
    panel.loc[panel["code"].isin(SELECTED_CODES) & panel["year"].between(START_YEAR, END_YEAR)]
         .copy()
         .assign(
             ln_co2=lambda d: np.log(d["co2_tonnes"]),
             ln_population=lambda d: np.log(d["population"]),
             ln_affluence=lambda d: np.log(d["gdp_per_capita_constant_2021_intl"]),
             year_centered=lambda d: d["year"] - d["year"].min(),
         )
)
print("shape:", analysis.shape)
print("countries:", sorted(analysis["code"].unique()))
display(analysis.head(3))


### 출력 해석: 원래 값과 로그 값

같은 행에서 원 단위 변수와 로그 변수가 어떻게 대응하는지 확인합니다.


In [ ]:
log_view = analysis[[
    "code", "year", "co2_tonnes", "ln_co2", "population", "ln_population",
    "gdp_per_capita_constant_2021_intl", "ln_affluence"
]].head(5)
print("로그를 취해도 행 수와 국가–연도 key는 바뀌지 않습니다.")
display(log_view)


## 2. 회귀명세의 단계별 비교

| 단계 | 추가되는 요소 | 주요 출력 |
|---:|---|---|
| 2-1 | 396행에 공통 회귀식을 적합한 pooled OLS | 적합값, 잔차, SSE, 계수 |
| 2-2 | 국가·연도 고정효과 | 동일 설명변수의 계수 변화 |
| 2-3 | HC3·국가 cluster 표준오차 | 점추정과 불확실성의 구분 |

모든 단계는 같은 결과변수와 설명변수를 사용합니다. Pooled와 고정효과의 차이는 계수를 추정할 때 활용하는 비교정보에 있으며, 표준오차 조정은 불확실성 계산에 적용됩니다.

> **확인 질문:** pooled OLS를 비교 기준으로 먼저 제시하는 이유는 무엇입니까?


### 2-1. Pooled OLS 기준선과 잔차

396개 국가–연도 행을 하나의 표본으로 결합해 공통 회귀식을 적합합니다. 실제 로그 CO₂와 적합값의 차이를 잔차로 정의하고, OLS가 최소화하는 잔차제곱합을 확인합니다.

$$e_i=y_i-\hat y_i, \qquad SSE=\sum_i e_i^2$$

이 단계에서는 고정효과와 표준오차 조정을 적용하지 않습니다. Pooled OLS는 이후 명세와 비교할 기준모형입니다.


In [ ]:
pooled_formula = "ln_co2 ~ ln_population + ln_affluence"
pooled_model = smf.ols(pooled_formula, data=analysis)
pooled_plain = pooled_model.fit()

pooled_preview = analysis[["code", "year", "ln_co2"]].copy()
pooled_preview["fitted"] = pooled_plain.fittedvalues
pooled_preview["residual"] = pooled_preview["ln_co2"] - pooled_preview["fitted"]
pooled_sse = float(np.square(pooled_preview["residual"]).sum())

print("[pooled OLS에 넣은 회귀식]")
print(pooled_formula)
print(f"관측 행 수: {int(pooled_plain.nobs)}")
print("\n[실제값 - 적합값 = 잔차]")
display(pooled_preview.head(8).round(4))
print(f"잔차 제곱합 SSE: {pooled_sse:.4f}")

pooled_point_table = pd.DataFrame({
    "term": ["ln_population", "ln_affluence"],
    "coefficient": [pooled_plain.params["ln_population"], pooled_plain.params["ln_affluence"]],
})
print("\n[pooled OLS 점추정 계수]")
display(pooled_point_table.round(4))


### 2-2. 국가·연도 고정효과와 비교정보

이번에는 설명변수는 그대로 두고 `C(code)`와 `C(year)`를 추가합니다.

- `C(code)`: 국가마다 다른 기본 높이를 허용해 시간에 거의 변하지 않는 국가 차이를 분리합니다.
- `C(year)`: 같은 해 여러 국가가 함께 경험한 공통 충격을 분리합니다.

두 명세의 점추정 계수 차이는 계수를 구성하는 비교정보가 달라졌음을 보여줍니다. 계수 차이만으로 어느 명세가 인과적으로 타당한지 판단할 수는 없습니다.


In [ ]:
fixed_effects_formula = "ln_co2 ~ ln_population + ln_affluence + C(code) + C(year)"
fixed_effects_model = smf.ols(fixed_effects_formula, data=analysis)
fixed_effects_plain = fixed_effects_model.fit()

terms = ["ln_population", "ln_affluence"]
point_estimate_table = pd.DataFrame({
    "term": terms,
    "pooled_coef": [pooled_plain.params[term] for term in terms],
    "fixed_effects_coef": [fixed_effects_plain.params[term] for term in terms],
})
point_estimate_table["difference_FE_minus_pooled"] = (
    point_estimate_table["fixed_effects_coef"] - point_estimate_table["pooled_coef"]
)

print("[두 모형의 점추정 계수 비교]")
display(point_estimate_table.round(4))
print("pooled는 국가 간 차이와 국가 내 변화를 함께 사용합니다.")
print("국가·연도 고정효과는 같은 국가 안의 변화가 더 직접적으로 반영되게 합니다.")
print("계수 차이만으로 어느 모형이 인과적으로 옳다고 판정하지 않습니다.")


### 출력 해석: 비교정보가 달라졌을 때의 계수 변화

같은 결과변수와 설명변수를 사용해도 국가 간 차이를 함께 쓰는 기준선과 국가·연도 차이를 분리한 모형에서 계수가 어떻게 달라지는지 확인합니다.


In [ ]:
coef_view = point_estimate_table.melt(
    id_vars="term",
    value_vars=["pooled_coef", "fixed_effects_coef"],
    var_name="specification",
    value_name="coefficient",
)
display(coef_view)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
fig, ax = plt.subplots(figsize=(7.5, 4.2))
sns.barplot(data=coef_view, x="term", y="coefficient", hue="specification", ax=ax)
ax.set(title="Same variable, different model specification", xlabel="Variable", ylabel="Estimated elasticity")
plt.tight_layout()
plt.show()


### 선택 심화 — 2-3. 표준오차 계산법에 따른 불확실성

`HC3`와 국가 단위 `cluster`는 같은 회귀식의 **점추정 계수**를 바꾸는 방법이 아니라 **표준오차 계산 방식**을 조정하는 방법입니다.

강의 표본은 국가가 12개뿐이므로 cluster 표준오차를 정밀한 결론처럼 읽지 않습니다. 독립 실습처럼 국가가 10개 미만이면 이 Notebook은 cluster 표준오차를 계산 결과표에 제시하지 않고 `NaN`으로 남깁니다.


In [ ]:
pooled_hc3 = pooled_model.fit(cov_type="HC3")
cluster_count = int(analysis["code"].nunique())

if cluster_count >= 10:
    fixed_effects_cluster = fixed_effects_model.fit(
        cov_type="cluster",
        cov_kwds={"groups": analysis["code"]},
    )
    fixed_effects = fixed_effects_cluster
    fixed_effects_se_values = [fixed_effects_cluster.bse[term] for term in terms]
    cluster_status = f"계산함 — 국가 cluster {cluster_count}개이므로 결과는 주의해서 해석"
else:
    fixed_effects_cluster = None
    fixed_effects = fixed_effects_plain
    fixed_effects_se_values = [np.nan for _ in terms]
    cluster_status = f"보고하지 않음 — 국가 cluster가 {cluster_count}개뿐임"

pooled = pooled_hc3
result_table = pd.DataFrame({
    "term": terms,
    "pooled_coef": [pooled_plain.params[term] for term in terms],
    "pooled_se_HC3": [pooled_hc3.bse[term] for term in terms],
    "fixed_effects_coef": [fixed_effects_plain.params[term] for term in terms],
    "fixed_effects_se_cluster": fixed_effects_se_values,
})
# 뒤의 진단·검토 셀이 사용하는 짧은 열 이름도 유지합니다.
result_table["pooled_se"] = result_table["pooled_se_HC3"]
result_table["fixed_effects_se"] = result_table["fixed_effects_se_cluster"]

uncertainty_table = pd.DataFrame({
    "term": terms,
    "pooled_default_se": [pooled_plain.bse[term] for term in terms],
    "pooled_HC3_se": [pooled_hc3.bse[term] for term in terms],
    "FE_default_se": [fixed_effects_plain.bse[term] for term in terms],
    "FE_country_cluster_se": fixed_effects_se_values,
})

print("[점추정 계수와 조정 표준오차]")
display(result_table[["term", "pooled_coef", "pooled_se_HC3", "fixed_effects_coef", "fixed_effects_se_cluster"]].round(4))
print("\n[표준오차 계산방식 비교]")
display(uncertainty_table.round(4))
print("cluster 표준오차:", cluster_status)
print("표준오차 조정 전후의 점추정 계수는 같습니다:",
      np.allclose(pooled_plain.params[terms], pooled_hc3.params[terms]))


## 3. 표본 구조와 잔차 진단

| 진단 | 확인 대상 | 해석 |
|---|---|---|
| coverage | 국가별 시작·종료연도와 행 수 | 균형 여부와 누락기간 |
| 설명변수 상관 | `ln_population`과 `ln_affluence` | 두 계수의 분리 추정 난이도 |
| fitted–residual | 적합값에 따른 잔차 패턴 | 비선형성·이분산·누락구조의 신호 |

잔차의 곡선, 깔때기 또는 국가별 패턴은 모형 명세를 추가로 검토할 근거입니다. 특정 관측의 제외는 데이터 오류나 사전에 정한 기준이 확인된 경우에 별도로 판단합니다.

> **확인 질문:** 잔차 그림에 뚜렷한 패턴이 없다는 사실만으로 모든 회귀가정이 충족되었다고 결론 내릴 수 있습니까?


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

diagnostics = analysis[["code", "year", "ln_population", "ln_affluence"]].copy()
diagnostics["fitted"] = fixed_effects_plain.fittedvalues
diagnostics["residual"] = fixed_effects_plain.resid

coverage = analysis.groupby("code").agg(first_year=("year", "min"), last_year=("year", "max"), rows=("year", "size"))
display(coverage)
print("correlation ln_population vs ln_affluence:", analysis[["ln_population", "ln_affluence"]].corr().iloc[0, 1])

sns.set_theme(style="whitegrid", context="notebook")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.scatterplot(data=diagnostics, x="fitted", y="residual", hue="code", alpha=0.75, ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Fixed-effects residual diagnostic", xlabel="Fitted log CO2", ylabel="Residual")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 4. 모형 비교 조건 검증

마지막 검사는 모형 비교에 필요한 조건이 유지되었는지 평가합니다.

- 12개 국가의 기간과 관측 수가 같은가
- 로그변수가 모두 유한한가
- pooled·고정효과 모형에서 동일한 핵심 계수가 추정되었는가
- 표준오차 조정 전후의 점추정 계수가 같은가
- cluster 수가 보고 기준을 충족하는가
- 잔차 수와 적합 관측 수가 일치하는가
- 두 명세가 같은 원자료·결과변수·설명변수를 사용했는가

검증 통과는 계산과 비교 조건의 일치를 뜻합니다. 계수의 인과적 해석이나 표본 밖 예측성능은 이 검사에 포함되지 않습니다.

> **확인 질문:** 이 검사가 확인하는 비교 조건과 별도 연구설계가 필요한 주장을 각각 하나씩 제시할 수 있습니까?


In [ ]:
checks = pd.Series({
    "selected countries present": set(analysis["code"]) == set(SELECTED_CODES),
    "balanced selected panel": coverage["rows"].nunique() == 1,
    "positive original variables": (analysis[["co2_tonnes", "population", "gdp_per_capita_constant_2021_intl"]] > 0).all().all(),
    "pooled residual identity": np.allclose(
        pooled_preview["ln_co2"] - pooled_preview["fitted"], pooled_preview["residual"]
    ),
    "core coefficients estimated": point_estimate_table[["pooled_coef", "fixed_effects_coef"]].notna().all().all(),
    "finite residuals": np.isfinite(diagnostics["residual"]).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("회귀 비교조건 검증: PASS")


## ChatGPT 저장 응답 검증: 회귀계수 문장의 조건 확인하기

점추정 계수표를 제공하고, 표본·기간·모형 명세와 탄력성 해석이 결과문장에 정확히 반영됐는지 검토합니다.

특히 통합 회귀(pooled OLS)와 국가·연도 고정효과가 사용하는 비교정보가 다르다는 점, 로그–로그 계수가 조건부 연관성을 나타낸다는 점을 실제 회귀표와 대조합니다.


In [ ]:
review_prompt = f"""
다음 환경 패널회귀의 점추정 계수표를 검토해 주세요.

- 국가: {sorted(analysis['code'].unique())}
- 기간: {analysis['year'].min()}~{analysis['year'].max()}
- 수업 명세: ln(CO2) ~ ln(population) + ln(affluence)
- 점추정 계수표:
{point_estimate_table.to_string(index=False)}

아래 순서로 답해 주세요.
1. pooled OLS가 섞어서 사용하는 국가 간 차이와 국가 내 변화
2. 국가·연도 고정효과가 주로 사용하는 같은 국가 안의 변화
3. 각 계수를 1% 변화 단위의 탄력성으로 읽은 문장
4. 두 명세 사이에서 계수가 얼마나 바뀌었는지와, 그 차이만으로 말할 수 없는 인과 주장

제약: 현재 제공된 표에는 표준오차와 p-value가 없습니다. 유의성이나 불확실성
수치를 만들지 말고, 결과문장에는 표본·기간·명세를 포함해 주세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 계수값을 실제 점추정 계수표에서 다시 찾고, 표에 없는 표준오차·p-value를 답변이 만들지 않았는지 확인합니다.
2. 문장에 표본·기간·명세와 탄력성 단위가 들어 있는지 확인합니다.
3. 고정효과를 사용했다는 이유만으로 인과효과라고 쓴 문장은 채택하지 않습니다.


## 해석 범위

회귀계수는 선택한 국가–연도 표본과 모형 명세에서 관찰된 조건부 연관성을 요약합니다.

국가·연도 고정효과를 포함해도 시간에 따라 변하는 누락요인, 역인과와 측정오차가 모두 제거되는 것은 아닙니다. 따라서 인구나 GDP의 독립적 인과효과 또는 정책효과로 해석하지 않으며, 사용한 표본·명세·표준오차 계산방식을 함께 제시합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

잔차와 OLS에서 출발해 국가–연도 패널의 비교정보를 구분하고, 통합 회귀와 고정효과 계수를 조건부 연관성으로 해석합니다.

### 적용 순서

1. `04_stirpat_regression_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `04_stirpat_regression_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 관측값, 적합값, 잔차는 어떤 관계이며 OLS는 왜 잔차제곱합(SSE)이 작은 회귀선을 선택합니까?
- pooled OLS는 국가 간 차이와 국가 내 시간 변화를 어떻게 섞으며, 국가·연도 고정효과는 비교의 기준을 어떻게 바꾸어 놓습니까?
- 로그–로그 계수를 탄력성으로 읽는 방법과, 그 계수를 인과효과로 단정할 수 없는 이유는 무엇입니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
